# Triple-Barrier Parameter Analysis

This notebook provides a workflow for analyzing and visualizing the choice of parameters for the Triple-Barrier labeling method.

The process is as follows:
1. **Load Execution Price Data**: Fetch the realistic bid/ask execution prices for a given market and date range.
2. **Calculate & Visualize Volatility**: Compute a rolling volatility measure from the historical data, which will be used to set dynamic barriers.
3. **Define Barriers & Generate Labels**: Use the calculated volatility to define the triple barriers and generate the resulting labels.
4. **Analyze Label Distribution**: Summarize and plot the distribution of outcomes (Win, Loss, Timeout) to understand the impact of the chosen parameters.

## Setup

Import libraries and define the core parameters for the analysis.

In [ ]:
import sys
sys.path.insert(0, '../src')

import logging
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from datetime import timedelta
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from training import config
from labeling.triple_barrier import (
    ts_index,
    calculate_volatility_and_spread,
    compute_labels,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
%matplotlib inline

SAVE_LABELS = True

(
    FEATURE_MARKET,
    LABEL_MARKET,
    EXECUTION_SIZE,
    TARGET_COLUMN,
    MIN_VOL_SPREAD_RATIO,
    CONFIG,
) = config.setup()

DATA_START_DATE = "20260102"
DATA_END_DATE = "20260105"
DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)

## 1. Load Execution Price Data

Load the execution price data which contains the realistic bid, ask, and mid prices for our chosen market and order size.

In [ ]:
read_range = DtRange(
    DT_RANGE.start_dt - timedelta(hours=8),
    DT_RANGE.end_dt + timedelta(hours=8),
)

print(f"Loading execution prices for {LABEL_MARKET.id} (size: {EXECUTION_SIZE})...")
price_store = ParquetStore(sys_env.get_parquet_root_dir()).create_execution_price(
    LABEL_MARKET, execution_size_str=EXECUTION_SIZE)

df = price_store.read(read_range)
display(df.shape, df.head())

df = ts_index(df)
# display(df.shape, df.head())

## 2. Calculate & Visualize Volatility

Here we implement **Methodology 1**: calculating dynamic barriers based on recent volatility. We'll compute the rolling standard deviation of 1-minute returns.

In [ ]:
df = calculate_volatility_and_spread(df, CONFIG)
# display(df.shape, df.head())

# print("Profit-take and Stop-loss levels (in %):")
# display((df[['pt_pct', 'sl_pct']] * 100).describe())

# MEMO to check if the result is healthy
# display(df[["volatility_ema", "spread_pct_ema"]].corr())  # expecting big value 0.6~
# display(df[["volatility_ema", "spread_pct_ema"]].diff().corr()) # expecting small value ~0.03

In [ ]:
# Plot the mid-price and the calculated volatility
fig, ax1 = plt.subplots(figsize=(15, 7))

color = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel('Spread', color=color)
ax1.plot(df.index, df['spread_pct_ema'], color=color, label='Spread')
ax1.tick_params(axis='y', labelcolor=color)

# ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
color = 'tab:red'
# ax2.set_ylabel('Volatility', color=color)
ax1.plot(df.index, df['volatility_ema'], color=color, alpha=0.7, label='Volatility')
# ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'{LABEL_MARKET.symbol.name} Spread and Volatility')
fig.tight_layout()
plt.show()

## 3. Define Barriers & Generate Labels

Now we use the calculated `volatility` series to define our dynamic barriers and generate labels. Each data point receives:
- A **label** (0=Short, 1=Long, 2=Timeout)
- **long_pnl**: PnL if you entered long (buy at ask, sell at bid at exit)
- **short_pnl**: PnL if you entered short (sell at bid, buy at ask at exit)

In [ ]:
print(f"Generating labels at tick level with {CONFIG.timeout_sec}s timeout...")
print(f"Data points: {len(df)}")
result = compute_labels(df, CONFIG)

labels_df = result.labels_df
print(f"Label count: {len(labels_df)}")

# Join labels back - already at tick level, direct assignment
df['label'] = labels_df['label']
df['long_pnl'] = labels_df['long_pnl']
df['short_pnl'] = labels_df['short_pnl']
print("Label generation complete.")
df.describe().T

## 4. Filter by Vol/Spread Ratio

Filter to keep only data points where volatility exceeds spread by the configured ratio.
This ensures we only analyze periods with favorable structural conditions for trading.
The filtering is done AFTER label generation so that labels are computed for all data points first.

In [ ]:
# Apply the vol/spread filter from TripleBarrierConfig
from training.plot_labels import plot_labels
from labeling.triple_barrier import filter_by_vol_spread_ratio

original_count = len(df)
plot_labels(df['label'], CONFIG)

filtered_df = filter_by_vol_spread_ratio(df, MIN_VOL_SPREAD_RATIO)
filtered_count = len(filtered_df)

plot_labels(filtered_df['label'], CONFIG)

print(f"Vol/Spread Filter (ratio={MIN_VOL_SPREAD_RATIO}):")
print(f"  Before: {original_count:,} rows")
print(f"  After: {filtered_count:,} rows")
print(f"  Removed: {original_count - filtered_count:,} "
      f"rows ({(original_count - filtered_count)/original_count*100:.1f}%)")

# Update labels_df for downstream analysis
labels_df = filtered_df[['label', 'long_pnl', 'short_pnl']].copy()
print(f"\nLabel distribution after filtering:")
print(labels_df['label'].value_counts().sort_index())

## 5. PnL Analysis with Realistic Execution

The `compute_labels` function calculates both long and short PnL for every data point:
- **long_pnl** = bid(exit) - ask(entry) — buy at ASK, sell at BID
- **short_pnl** = bid(entry) - ask(exit) — sell at BID, buy at ASK

This allows realistic simulation for both long and short positions.

In [ ]:
# PnL analysis using long_pnl and short_pnl

# Exclude edge effect (last timeout_sec) for valid analysis
cutoff_time = labels_df.index.max() - pd.Timedelta(seconds=CONFIG.timeout_sec)
pnl_df = labels_df.loc[labels_df.index <= cutoff_time].copy()

# Calculate pnl in basis points using average mid price as reference
avg_mid = df['mid'].mean()
pnl_df['long_pnl_bps'] = (pnl_df['long_pnl'] / avg_mid) * 10000
pnl_df['short_pnl_bps'] = (pnl_df['short_pnl'] / avg_mid) * 10000

print(f"Total samples: {len(pnl_df):,}")
print(f"\n{'='*60}")
print("PnL Summary by Label Type (0=Short, 1=Long, 2=Timeout)")
print('='*60)

label_names = {0: 'Short (label=0)', 1: 'Long (label=1)', 2: 'Timeout (label=2)'}
for label_val in [1, 0, 2]:
    subset = pnl_df[pnl_df['label'] == label_val]
    if len(subset) == 0:
        continue
    print(f"\n{label_names[label_val]}: {len(subset):,} samples")
    print(f"  Long PnL:  avg={subset['long_pnl'].mean():.4f} ({subset['long_pnl_bps'].mean():.2f} bps), total={subset['long_pnl'].sum():.4f}")
    print(f"  Short PnL: avg={subset['short_pnl'].mean():.4f} ({subset['short_pnl_bps'].mean():.2f} bps), total={subset['short_pnl'].sum():.4f}")

# Overall summary
print(f"\n{'='*60}")
print("Overall Summary")
print('='*60)
print(f"Total Long PnL (always-long): {result.total_long_pnl:.4f}")
print(f"Avg Long PnL per sample: {result.avg_long_pnl:.6f}")
print(f"Total Short PnL (always-short): {pnl_df['short_pnl'].sum():.4f}")
print(f"Avg Short PnL per sample: {pnl_df['short_pnl'].mean():.6f}")
print(f"Long win rate (long_pnl > 0): {(pnl_df['long_pnl'] > 0).mean():.2%}")
print(f"Short win rate (short_pnl > 0): {(pnl_df['short_pnl'] > 0).mean():.2%}")

# Show the labels_df with pnl
display(labels_df.head())

In [ ]:
# Visualize PnL distribution (long_pnl and short_pnl)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Filter out rows with zero pnl in both directions
pnl_valid = pnl_df[(pnl_df['long_pnl'] != 0) | (pnl_df['short_pnl'] != 0)].copy()

label_info = [
    (1, 'green', 'Long (label=1)'),
    (0, 'red', 'Short (label=0)'),
    (2, 'orange', 'Timeout (label=2)'),
]

# 1. Long PnL distribution by label
ax1 = axes[0, 0]
for label_val, color, name in label_info:
    subset = pnl_valid[pnl_valid['label'] == label_val]['long_pnl_bps']
    if len(subset) > 0:
        ax1.hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset):,})')
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax1.set_xlabel('Long PnL (bps)')
ax1.set_ylabel('Count')
ax1.set_title('Long PnL Distribution by Label')
ax1.legend()

# 2. Short PnL distribution by label
ax2 = axes[0, 1]
for label_val, color, name in label_info:
    subset = pnl_valid[pnl_valid['label'] == label_val]['short_pnl_bps']
    if len(subset) > 0:
        ax2.hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset):,})')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Short PnL (bps)')
ax2.set_ylabel('Count')
ax2.set_title('Short PnL Distribution by Label')
ax2.legend()

# 3. Cumulative PnL over time (always-long vs always-short)
ax3 = axes[1, 0]
pnl_sorted = pnl_valid.sort_index()
ax3.plot(pnl_sorted.index, pnl_sorted['long_pnl'].cumsum(), linewidth=1, label='Always Long', color='green')
ax3.plot(pnl_sorted.index, pnl_sorted['short_pnl'].cumsum(), linewidth=1, label='Always Short', color='red')
ax3.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative PnL')
ax3.set_title('Cumulative PnL Over Time')
ax3.legend()
ax3.tick_params(axis='x', rotation=45)

# 4. Average PnL by label (bar chart - both directions)
ax4 = axes[1, 1]
label_names_short = {1: 'Long', 0: 'Short', 2: 'Timeout'}
labels_present = [l for l in [1, 0, 2] if l in pnl_valid['label'].values]
x = range(len(labels_present))
width = 0.35

long_avgs = [pnl_valid[pnl_valid['label'] == l]['long_pnl_bps'].mean() for l in labels_present]
short_avgs = [pnl_valid[pnl_valid['label'] == l]['short_pnl_bps'].mean() for l in labels_present]

bars1 = ax4.bar([i - width/2 for i in x], long_avgs, width, label='Long PnL', color='green', alpha=0.7)
bars2 = ax4.bar([i + width/2 for i in x], short_avgs, width, label='Short PnL', color='red', alpha=0.7)
ax4.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax4.set_xlabel('Label')
ax4.set_ylabel('Average PnL (bps)')
ax4.set_title('Average PnL by Label and Direction')
ax4.set_xticks(list(x))
ax4.set_xticklabels([label_names_short[l] for l in labels_present])
ax4.legend()

plt.tight_layout()
plt.show()

# Show detailed statistics
print("\nDetailed Long PnL Statistics by Label (in bps):")
display(pnl_valid.groupby('label')['long_pnl_bps'].describe().round(2))
print("\nDetailed Short PnL Statistics by Label (in bps):")
display(pnl_valid.groupby('label')['short_pnl_bps'].describe().round(2))

## 6. Save label data

In [ ]:
if SAVE_LABELS:
    # Save to tmp parquet store
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir()).create_label(
        "TripleBarrierLabel_tmp",
        LABEL_MARKET,
        CONFIG.param_id,
    )
    parquet_store.add_dataframe(df.copy(), ts_column=None)
    parquet_store.close()
    print(f"GenerateLabelsStep: Saved {len(df)} labels to Parquet " f"(param_id={CONFIG.param_id})")

# Memo

### What affects the win/loss rate

Setup
- Resample interval of volatility calculation = Tvol
- Timeout = Tvol
- Target profit = P

Factors
- Volatility = V
- Spread = S

```
Loss rate = 0.5 + 0.5 * (S/V)
```

The "Gambler's Ruin" Analogy

Imagine a particle starting at position 0 on a number line. At each step, it moves +1 or -1 with a 50% chance (a "symmetric random walk"). The game stops if the particle hits one of two walls:
* An upper "win" wall at position +a.
* A lower "loss" wall at position -b.

The well-known solution for the probability of hitting the lower wall at -b before hitting the upper wall at +a is:

P(Loss) = a / (a + b)

Applying it to Our Trading Model

Now, let's map our trading variables to this model:
* The "particle" is the true underlying price of the asset.
* a is the distance the price needs to travel up to trigger a win. As we established, this is v + s (it has to cover the barrier distance v plus the spread s).
* b is the distance the price needs to travel down to trigger a loss. This is v - s.

Now, we just plug our financial model's distances into the classic Gambler's Ruin formula:

1. Start with the formula:
P(Loss) = a / (a + b)

2. Substitute our distances:
P(Loss) = (v + s) / ( (v + s) + (v - s) )

3. Simplify the denominator:
P(Loss) = (v + s) / (2v)

4. Split the fraction:
P(Loss) = v/(2v) + s/(2v)

5. And you get the final expression:
P(Loss) = 0.5 + 0.5 * (s/v)

### The process/purpose to execute the notebook

Summary: A Dynamic Labeling Strategy Based on Structural Profitability

The proposed methodology is to evolve from a simple, static labeling process to a dynamic, regime-aware filtering system. The core principle is to only generate labels for entry points that have a favorable
structural probability of success, based on real-time market conditions.

This is achieved through the following steps:

1. Modeling Market Conditions: We will continuously track two key metrics:
* Transaction Cost: Represented by an Exponential Moving Average (EMA) of the bid-ask spread (s).
* Market Volatility: Represented by an EMA of the price volatility (v), calculated over a time horizon appropriate for the strategy.

2. Dynamic Entry Filtering: At each moment, we will use the formula derived from the "Gambler's Ruin" problem—`P(Loss) ≈ 0.5 + 0.5 * (s/v)`—to calculate the theoretical probability of a loss for a potential
trade initiated under the current market conditions.

3. Applying a Quality Threshold: We will define a hyperparameter, `X`, which represents the maximum acceptable theoretical loss rate. A triple-barrier label will only be generated if the calculated P(Loss) is
less than X. This acts as a filter to discard trades initiated during unfavorable regimes (e.g., high-spread/low-volatility periods).

4. Time Horizon Alignment: The volatility measurement window must be carefully tuned to align with the desired trading frequency (e.g., shorter windows for scalping strategies) to ensure the s/v ratio is
meaningful.

In essence, you are proposing to build a meta-model that pre-qualifies trades. It uses a simple, powerful heuristic to decide when the game is even worth playing, ensuring that the final labels are generated only
for high-quality opportunities. This is a superb approach that should dramatically increase the robustness and value of the resulting labels.


## 99. Pre-Labeling Filter: Structural Profitability Analysis

Before running the computationally expensive labeling process, we'll analyze the structural profitability of the market over the selected period. We use the 'Gambler's Ruin' heuristic `P(Loss) ≈ 0.5 + 0.5 * (s/v)` to estimate the probability of loss based on the current relationship between transaction costs (spread `s`) and volatility (`v`).

If the average theoretical loss rate is higher than our defined `MAX_THEORETICAL_LOSS_RATE`, it suggests the market conditions are structurally unfavorable for our strategy's parameters, and we should consider adjusting them before proceeding.

In [ ]:
# To make spread and volatility comparable, we need to express spread as a percentage of price
# s_pct = spread / mid_price
# v_pct = BARRIER_MULTIPLIER * volatility
# P(Loss) = 0.5 + 0.5 * (s_pct / v_pct)

# Use the smoothed EMA values for a more stable calculation
# Parameters for the new structural profitability filter

MAX_THEORETICAL_LOSS_RATE = 0.65 # Set a target max loss rate for filtering (e.g., 0.65 = 65%)

s_pct_ema = df['spread_pct_ema']
v_pct_ema = df['volatility_ema']

# Calculate the theoretical probability of loss for each data point
# We cap the ratio at a high number to avoid extreme values from near-zero volatility
s_div_v = (s_pct_ema / v_pct_ema).clip(upper=5.0)
df['theoretical_loss_p'] = 0.5 + 0.5 * s_div_v

avg_loss_p = df['theoretical_loss_p'].mean()
avg_spread_ema = s_pct_ema.mean()
avg_volatility_ema = v_pct_ema.mean()

print("--- Structural Profitability Analysis ---")
print(f"Average EMA Spread: {avg_spread_ema:.5f}")
print(f"Average EMA Volatility (absolute): {avg_volatility_ema:.5f}")
print(f"Average Theoretical Loss Rate: {avg_loss_p:.2%}")
print(f"Your Maximum Acceptable Rate: {MAX_THEORETICAL_LOSS_RATE:.2%}")